In [ ]:
from pathlib import Path

for p in Path("/kaggle/input/notebooks/urajkaushik").iterdir():
    print(p)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/input/notebooks/urajkaushik/notebook191c8a3bcd")
DATA = ROOT / "cbs-data"

if not ROOT.exists():
    raise FileNotFoundError(f"Mounted notebook output not found: {ROOT}")

print("Notebook output:", ROOT)
print("cbs-data folder exists:", DATA.exists())

raw = DATA / "raw"
print("Raw FITS files:", len(list(raw.rglob("*.fits"))) if raw.exists() else "folder missing")

files_to_check = [
    "manifest.csv",
    "features.csv",
    "candidate_consensus.csv",
    "next_candidates_bls.csv",
    "cosmic_swan_candidate_review.csv",
    "cosmic_swan_final_review.csv",
    "cosmic_swan_follow_up_shortlist.csv",
    "cosmic_swan_follow_up_visual_review.csv",
    "blindspot_sensitivity_map.csv",
    "injection_recovery_rows.csv",
    "injection_recovery_run.json",
    "injection_recovery_summary.csv",
    "run.json",
    "TIC_229137615.json",
    "cosmic_swan_screening_pilot_report.md",
]

for name in files_to_check:
    matches = list(ROOT.rglob(name))
    if not matches:
        print(f"MISSING: {name}")
        continue
    for path in matches:
        if path.suffix == ".csv":
            try:
                print(f"{name}: {len(pd.read_csv(path)):,} rows — {path}")
            except Exception as e:
                print(f"{name}: found, but couldn't read CSV ({e}) — {path}")
        else:
            print(f"FOUND: {path}")

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/kaggle/input/notebooks/urajkaushik/notebook191c8a3bcd")
DATA = ROOT / "cbs-data"

def read_csv(name, paths):
    path = next((p for p in paths if p.exists() and p.stat().st_size > 0), None)
    if path is None:
        print(f"{name}: missing or empty")
        return None
    df = pd.read_csv(path)
    print(f"\n{name}: {len(df):,} rows")
    print("Columns:", list(df.columns))
    for col in ["source_id", "tic_id", "TIC", "observation_id", "sector"]:
        if col in df.columns:
            print(f"  unique {col}: {df[col].nunique(dropna=True):,}")
    if "sector" in df.columns:
        print("  rows by sector:")
        print(df.groupby("sector").size().to_string())
    return df

features = read_csv("Features", [DATA / "features.csv"])
consensus = read_csv("Consensus", [ROOT / "candidate_consensus.csv"])

fits = list((DATA / "raw").rglob("*.fits"))
print(f"\nSaved raw FITS files: {len(fits)}")
for f in fits:
    print(" ", f.name)

series = DATA / "series.npy"
print("\nseries.npy exists:", series.exists())

for name in ["run.json", "injection_recovery_run.json"]:
    path = DATA / "scores" / name
    print(f"\n{name} exists:", path.exists())
    if path.exists():
        info = json.loads(path.read_text())
        print(json.dumps(info, indent=2)[:2500])

In [ ]:
from pathlib import Path
import numpy as np

base = Path("/kaggle/input")
data = Path("/kaggle/input/notebooks/urajkaushik/notebook191c8a3bcd/cbs-data")

print("Attached inputs:")
for path in base.iterdir():
    print(" ", path)

fits = list(base.rglob("*.fits")) + list(base.rglob("*.fits.gz"))
print("\nFITS files across attached inputs:", len(fits))
print("Light-curve FITS:", sum("_lc.fits" in p.name.lower() for p in fits))
print("Target-pixel FITS:", sum("_tp.fits" in p.name.lower() for p in fits))

print("\nZIP files:")
for path in base.rglob("*.zip"):
    print(" ", path)

print("\nManifests:")
for path in base.rglob("manifest.csv"):
    print(" ", path)

series = data / "series.npy"
print("\nseries.npy:", series.exists(),
      f"({series.stat().st_size:,} bytes)" if series.exists() else "")
if series.exists():
    try:
        arr = np.load(series, mmap_mode="r", allow_pickle=False)
        print("Series shape and dtype:", arr.shape, arr.dtype)
    except Exception as e:
        print("Could not inspect array without pickle:", e)

In [ ]:
from pathlib import Path
import pandas as pd

parent = Path("/kaggle/input/notebooks/urajkaushik")

for name in ["notebook191c8a3bcd", "cosmic-swan-4"]:
    root = parent / name
    data = root / "cbs-data"

    print(f"\n--- {name} ---")
    print("Notebook output found:", root.exists())

    if not root.exists():
        continue

    manifest = data / "manifest.csv"
    if manifest.exists():
        df = pd.read_csv(manifest)
        print("Manifest rows:", len(df))
        print("Manifest columns:", list(df.columns))
        if "status" in df.columns:
            print("Statuses:", df["status"].value_counts(dropna=False).to_dict())
    else:
        print("Manifest: missing")

    raw = data / "raw"
    fits = list(raw.rglob("*.fits")) if raw.exists() else []
    print("Raw FITS files:", len(fits))
    print("Light-curve FITS:", sum("_lc.fits" in p.name.lower() for p in fits))
    print("Target-pixel FITS:", sum("_tp.fits" in p.name.lower() for p in fits))

    features = data / "features.csv"
    print("Features rows:", len(pd.read_csv(features)) if features.exists() else "missing")

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

old = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
work = Path("/kaggle/working/cbs-data")
raw = old / "raw"

assert (old / "manifest.csv").exists(), "Cosmic Swan 4 manifest is missing"
assert (old / "features.csv").exists(), "Cosmic Swan 4 features are missing"
assert (old / "series.npy").exists(), "Cosmic Swan 4 series are missing"

if work.exists() and any(work.iterdir()):
    raise RuntimeError(
        f"{work} already has files. Stop and show me its contents; don't overwrite it."
    )

# Map the saved manifest and feature rows to the attached read-only FITS files.
raw_by_name = {p.name: p for p in raw.rglob("*.fits")}
manifest = pd.read_csv(old / "manifest.csv", dtype={"source_id": "string"})
features = pd.read_csv(old / "features.csv", dtype={"source_id": "string"})

def attached_fits(filename):
    name = Path(str(filename)).name
    if name not in raw_by_name:
        raise FileNotFoundError(f"Could not find attached FITS file: {name}")
    return str(raw_by_name[name])

manifest["local_path"] = manifest["product_filename"].map(attached_fits)
features["path"] = features["path"].map(attached_fits)

work.mkdir(parents=True)
manifest.to_csv(work / "manifest.csv", index=False)
features.to_csv(work / "features.csv", index=False)
shutil.copy2(old / "series.npy", work / "series.npy")

print("Working baseline prepared:", work)
print("Manifest rows:", len(manifest))
print("Manifest statuses:", manifest["status"].value_counts().to_dict())
print("Feature rows:", len(features))
print("Series file copied:", (work / "series.npy").stat().st_size, "bytes")
print("Old FITS files remain read-only under:", raw)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import pandas as pd

data_root = Path("/kaggle/working/cbs-data")
manifest_path = data_root / "manifest.csv"

# The baseline-preparation cell must run first.
assert manifest_path.exists(), "Run the baseline-preparation cell first."
manifest = pd.read_csv(manifest_path)
complete_count = manifest["status"].isin(["COMPLETE", "ARCHIVED"]).sum()
assert 5000 <= complete_count <= 6000, (
    f"Expected 5,000–6,000 complete files; found {complete_count}."
)

# Copy and install the project in Kaggle's writable area.
source_project = Path(
    "/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-project-src"
)
project = Path("/kaggle/working/cbs-project-src")
assert (source_project / "pyproject.toml").exists(), (
    f"Attached project source not found: {source_project}"
)
shutil.copytree(source_project, project, dirs_exist_ok=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(project)],
    check=True,
)

# Fetch up to 100 additional files per sector, stopping at 6,000 total.
failed_sectors = []

for sector in range(1, 11):
    manifest = pd.read_csv(manifest_path)
    complete_count = manifest["status"].isin(["COMPLETE", "ARCHIVED"]).sum()

    if complete_count >= 6000:
        print(f"Target reached: {complete_count} complete files.")
        break

    print(f"\nFetching sector {sector}; currently {complete_count}/6000")
    try:
        subprocess.run(
            [
                "cbs", "fetch",
                "--data-root", str(data_root),
                "--sectors", str(sector),
                "--max-lightcurves", "6000",
                "--per-sector-cap", "100",
                "--batch-size", "25",
            ],
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        failed_sectors.append(sector)
        print(f"Sector {sector} failed; continuing. Exit code: {exc.returncode}")

manifest = pd.read_csv(manifest_path)
complete = manifest[manifest["status"].isin(["COMPLETE", "ARCHIVED"])]

print("\nComplete files:", len(complete))
print("Complete files per sector:")
print(complete.groupby("sector").size().sort_index())
print("Failed sectors:", failed_sectors)

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

data_root = Path("/kaggle/working/cbs-data")

source_candidates = [
    Path("/kaggle/working/cbs-project-src/src"),
    Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-project-src/src"),
    Path("/kaggle/input/notebooks/urajkaushik/notebook191c8a3bcd/cbs-project-src/src"),
]
project_src = next(
    (p for p in source_candidates if (p / "cosmic_black_swan" / "score.py").exists()),
    None,
)
if project_src is None:
    raise FileNotFoundError("Could not find the project source.")

manifest = pd.read_csv(data_root / "manifest.csv")
complete_count = manifest["status"].isin(["COMPLETE", "ARCHIVED"]).sum()
features = pd.read_csv(data_root / "features.csv")
series = np.load(data_root / "series.npy", mmap_mode="r")

print("Complete files:", complete_count)
print("Feature rows:", len(features))
print("Series shape:", series.shape)

assert complete_count == 6000, "Expected 6,000 complete light curves."
assert len(features) == 6000 and series.shape == (6000, 512), (
    "Features or series are not yet at the expected 6,000-row size."
)

sys.path.insert(0, str(project_src))
from cosmic_black_swan.score import score_isolation_forest

result = score_isolation_forest(data_root, contamination=0.02, seed=42)
print(json.dumps(result, indent=2))